In [ ]:
# inspect canonical Week-5 features and labels

import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objs as go

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

ROOT = Path("../")
DATA_DERIVED = ROOT / "data" / "derived"

paths = {
    "features": DATA_DERIVED / "mru_usd_features_all_week5.csv",
    "L1_Q": DATA_DERIVED / "mru_usd_labels_L1_Q_k5_q20_60_20.csv",
    "L1_extreme": DATA_DERIVED / "mru_usd_labels_L1_k5_l075.csv",
    "L2": DATA_DERIVED / "mru_usd_labels_L2_ma10_50_h5.csv",
}

print("=== Paths being used (B1.1) ===")
for name, p in paths.items():
    print(f"{name:10s}: {p}  (exists={p.exists()})")

dfs = {}
for name, p in paths.items():
    print("\n" + "=" * 80)
    print(f"Loading '{name}' from: {p}")
    df = pd.read_csv(p)
    dfs[name] = df

    print(f"\n[name={name}] shape: {df.shape}")
    print(f"[name={name}] columns:")
    print(list(df.columns))

    print(f"\n[name={name}] dtypes:")
    print(df.dtypes)

    print(f"\n[name={name}] head(5):")
    print(df.head(5))

    date_like_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
    print(f"\n[name={name}] potential date/time columns: {date_like_cols}")

    print(f"\n[name={name}] NaN counts (first up to 20 columns):")
    nan_counts = df.isna().sum()
    print(nan_counts.head(20))



=== Paths being used (B1.1) ===
features  : ..\data\derived\mru_usd_features_all_week5.csv  (exists=True)
L1_Q      : ..\data\derived\mru_usd_labels_L1_Q_k5_q20_60_20.csv  (exists=True)
L1_extreme: ..\data\derived\mru_usd_labels_L1_k5_l075.csv  (exists=True)
L2        : ..\data\derived\mru_usd_labels_L2_ma10_50_h5.csv  (exists=True)

Loading 'features' from: ..\data\derived\mru_usd_features_all_week5.csv

[name=features] shape: (6752, 15)
[name=features] columns:
['date', 'open', 'high', 'low', 'close', 'log_price', 'log_return', 'f_r_lag1', 'f_past_ret_5', 'f_past_ret_20', 'f_vol_10', 'f_vol_30', 'f_ma_50', 'f_dist_50', 'f_high_vol']

[name=features] dtypes:
date              object
open             float64
high             float64
low              float64
close            float64
log_price        float64
log_return       float64
f_r_lag1         float64
f_past_ret_5     float64
f_past_ret_20    float64
f_vol_10         float64
f_vol_30         float64
f_ma_50          float64
f_dist_

In [ ]:


import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objs as go

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)

ROOT = Path("../")
DATA_DERIVED = ROOT / "data" / "derived"

paths = {
    "features": DATA_DERIVED / "mru_usd_features_all_week5.csv",
    "L1_Q": DATA_DERIVED / "mru_usd_labels_L1_Q_k5_q20_60_20.csv",
    "L1_extreme": DATA_DERIVED / "mru_usd_labels_L1_k5_l075.csv",
    "L2": DATA_DERIVED / "mru_usd_labels_L2_ma10_50_h5.csv",
}

print("=== B1.2 – Reloading CSVs ===")
dfs = {}
for name, p in paths.items():
    df = pd.read_csv(p)
    dfs[name] = df
    print(f"{name:10s}: shape={df.shape}")


for name, df in dfs.items():
    # Convert date to datetime
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
    df.sort_values("date", inplace=True)
    df.reset_index(drop=True, inplace=True)

    print("\n" + "=" * 80)
    print(f"[{name}] after datetime conversion & sort")
    print(f"  shape        : {df.shape}")
    print(f"  date min/max : {df['date'].min()}  ->  {df['date'].max()}")
    # Check monotonicity
    is_monotonic = df["date"].is_monotonic_increasing
    print(f"  date monotonic increasing? {is_monotonic}")


feat_dates = dfs["features"]["date"]

print("\n" + "=" * 80)
print("Checking date alignment across all DataFrames (vs 'features')")

for name, df in dfs.items():
    same_len = len(df) == len(feat_dates)
    same_all = np.array_equal(df["date"].values, feat_dates.values)
    print(f"[compare features vs {name:10s}] same_len={same_len}, same_all={same_all}")

    if not same_all:
        set_diff_1 = set(feat_dates) - set(df["date"])
        set_diff_2 = set(df["date"]) - set(feat_dates)
        print(f"  dates in features but not in {name}: {len(set_diff_1)}")
        print(f"  dates in {name} but not in features: {len(set_diff_2)}")


features_df = dfs["features"].copy()

l1_q_df = dfs["L1_Q"][["date", "R_t_to_tk", "y_L1_Q"]].copy()
l1_ext_df = dfs["L1_extreme"][["date", "R_t_to_tk", "y_L1"]].copy()
l2_df = dfs["L2"][["date", "MA_fast", "MA_slow", "MA_slow_slope", "y_L2"]].copy()

# For integrity: check that R_t_to_tk from L1_Q and L1_extreme are identical
r_equal = np.allclose(
    l1_q_df["R_t_to_tk"].fillna(0.0).values,
    l1_ext_df["R_t_to_tk"].fillna(0.0).values,
    atol=1e-12
)
print("\n" + "=" * 80)
print("Integrity check: R_t_to_tk from L1_Q vs L1_extreme")
if not r_equal:
    diff = (l1_q_df["R_t_to_tk"] - l1_ext_df["R_t_to_tk"]).abs()
    print("  max abs difference:", diff.max())
else:
    print("  OK ")

# Drop the duplicate R_t_to_tk from the L1_extreme side
l1_ext_df = l1_ext_df.drop(columns=["R_t_to_tk"])

merged = features_df.merge(l1_q_df, on="date", how="left", validate="one_to_one")
merged = merged.merge(l1_ext_df, on="date", how="left", validate="one_to_one")
merged = merged.merge(l2_df, on="date", how="left", validate="one_to_one")

print("\n" + "=" * 80)
print("Merged DataFrame summary")
print(f"shape: {merged.shape}")
print("columns:")
print(list(merged.columns))

print("\nHead(5):")
print(merged.head(5))

print("\nTail(5):")
print(merged.tail(5))

label_cols = ["R_t_to_tk", "y_L1_Q", "y_L1", "y_L2"]
print("\nNaN counts for key label columns:")
print(merged[label_cols].isna().sum())

for col in ["y_L1_Q", "y_L1", "y_L2"]:
    print("\n" + "-" * 40)
    print(f"Value counts for {col} (including NaNs):")
    print(merged[col].value_counts(dropna=False))




=== B1.2 – Reloading CSVs ===
features  : shape=(6752, 15)
L1_Q      : shape=(6752, 5)
L1_extreme: shape=(6752, 5)
L2        : shape=(6752, 6)

[features] after datetime conversion & sort
  shape        : (6752, 15)
  date min/max : 2000-01-03 00:00:00  ->  2025-11-25 00:00:00
  date monotonic increasing? True

[L1_Q] after datetime conversion & sort
  shape        : (6752, 5)
  date min/max : 2000-01-03 00:00:00  ->  2025-11-25 00:00:00
  date monotonic increasing? True

[L1_extreme] after datetime conversion & sort
  shape        : (6752, 5)
  date min/max : 2000-01-03 00:00:00  ->  2025-11-25 00:00:00
  date monotonic increasing? True

[L2] after datetime conversion & sort
  shape        : (6752, 6)
  date min/max : 2000-01-03 00:00:00  ->  2025-11-25 00:00:00
  date monotonic increasing? True

Checking date alignment across all DataFrames (vs 'features')
[compare features vs features  ] same_len=True, same_all=True
[compare features vs L1_Q      ] same_len=True, same_all=True
[comp

In [ ]:
#Config-driven loader for merged modeling DataFrame

import pandas as pd
import numpy as np
from pathlib import Path
import yaml

import plotly.express as px
import plotly.graph_objs as go

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 80)

ROOT = Path("../")
CONFIG_PATH = ROOT / "config" / "labels_features.yaml"


def load_labels_features_config(config_path: Path = CONFIG_PATH) -> dict:

    print(f"Loading config from: {config_path}")
    with config_path.open("r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    # Basic sanity checks
    required_top = ["dataset", "labels", "features", "defaults"]
    missing = [k for k in required_top if k not in config]
    if missing:
        print(f"WARNING: missing top-level keys in config: {missing}")
    else:
        print("Config top-level keys present:", list(config.keys()))

    print("Available labels in config:", list(config.get("labels", {}).keys()))
    print("Available feature sets in config:", list(config.get("features", {}).keys()))

    return config


def build_modeling_dataframe(
    config: dict,
    labels_to_include=("L1_Q", "L1_extreme", "L2"),
    include_l2_rule_components: bool = True,
) -> pd.DataFrame:
    """
    Build merged modeling DataFrame from config-defined feature and label files.

    Parameters
    ----------
    config : dict
        Parsed labels_features configuration.
    labels_to_include : tuple of str
        Which label keys from config['labels'] to include.
    include_l2_rule_components : bool
       include MA_fast, MA_slow, MA_slow_slope from the L2 label file.

    Returns
    -------
    merged : pd.DataFrame
        DataFrame with features + selected labels merged on date.
    """
    dataset_cfg = config["dataset"]
    labels_cfg = config["labels"]
    features_cfg = config["features"]

    features_file = ROOT / Path(dataset_cfg["features_file"])
    print(f"\n[build_modeling_dataframe] Loading features from: {features_file}")
    feat_df = pd.read_csv(features_file)
    print(f"  features shape: {feat_df.shape}")

    feat_df["date"] = pd.to_datetime(feat_df["date"], format="%Y-%m-%d")
    feat_df.sort_values("date", inplace=True)
    feat_df.reset_index(drop=True, inplace=True)

    merged = feat_df.copy()

    # Load and merge each label
    for lbl_key in labels_to_include:
        if lbl_key not in labels_cfg:
            print(f"WARNING: label '{lbl_key}' not found in config['labels'];")
            continue

        lbl_info = labels_cfg[lbl_key]
        lbl_file = ROOT / Path(lbl_info["file"])
        lbl_col = lbl_info["label_column"]

        print(f"\n  Loading label '{lbl_key}' from: {lbl_file}")
        lbl_df = pd.read_csv(lbl_file)
        print(f"    raw shape: {lbl_df.shape}")
        print(f"    columns: {list(lbl_df.columns)}")

        # Convert date to datetime and sort
        lbl_df["date"] = pd.to_datetime(lbl_df["date"], format="%Y-%m-%d")
        lbl_df.sort_values("date", inplace=True)
        lbl_df.reset_index(drop=True, inplace=True)

        # Decide which columns to keep from this label file
        keep_cols = ["date", lbl_col]
        # For L1_Q we also want the canonical R_t_to_tk
        if lbl_key == "L1_Q":
            if "future_return_column" in lbl_info:
                fr_col = lbl_info["future_return_column"]
                if fr_col in lbl_df.columns:
                    keep_cols.append(fr_col)
        if lbl_key == "L2" and include_l2_rule_components:
            for c in ["MA_fast", "MA_slow", "MA_slow_slope"]:
                if c in lbl_df.columns:
                    keep_cols.append(c)

        lbl_df = lbl_df[keep_cols].copy()

        # Integrity check: dates must match
        same_len = len(lbl_df) == len(merged)
        same_dates = np.array_equal(lbl_df["date"].values, merged["date"].values)
        print(f"    after date sort: shape={lbl_df.shape}, same_len={same_len}, same_dates={same_dates}")
        if not same_dates:
            print("    WARNING: date misalignment detected when merging labels")

        # Merge on date
        merged = merged.merge(lbl_df, on="date", how="left", validate="one_to_one")

    print("\n[build_modeling_dataframe] Final merged shape:", merged.shape)
    print("Merged columns:")
    print(list(merged.columns))

    label_cols_present = [c for c in ["R_t_to_tk", "y_L1_Q", "y_L1", "y_L2"] if c in merged.columns]
    if label_cols_present:
        print("\nNaN counts for label-related columns:")
        print(merged[label_cols_present].isna().sum())
    else:
        print("\nNo known label columns present in merged DataFrame.")

    return merged



config = load_labels_features_config(CONFIG_PATH)
modeling_df = build_modeling_dataframe(config)

print("\nHead(5) of modeling_df:")
print(modeling_df.head(5))

print("\nTail(5) of modeling_df:")
print(modeling_df.tail(5))

print("\nBasic label distributions from modeling_df:")
for col in ["y_L1_Q", "y_L1", "y_L2"]:
    if col in modeling_df.columns:
        print("\n" + "-" * 40)
        print(f"Value counts for {col} (including NaNs):")
        print(modeling_df[col].value_counts(dropna=False))



Loading config from: ..\config\labels_features.yaml
Config top-level keys present: ['dataset', 'validation', 'labels', 'features', 'defaults']
Available labels in config: ['L1_Q', 'L1_extreme', 'L2']
Available feature sets in config: ['source_file', 'F_CORE', 'F_MIN', 'optional', 'anti_circularity', 'forbidden_for_L2']

[build_modeling_dataframe] Loading features from: ..\data\derived\mru_usd_features_all_week5.csv
  features shape: (6752, 15)

  Loading label 'L1_Q' from: ..\data\derived\mru_usd_labels_L1_Q_k5_q20_60_20.csv
    raw shape: (6752, 5)
    columns: ['date', 'close', 'log_return', 'R_t_to_tk', 'y_L1_Q']
    after date sort: shape=(6752, 3), same_len=True, same_dates=True

  Loading label 'L1_extreme' from: ..\data\derived\mru_usd_labels_L1_k5_l075.csv
    raw shape: (6752, 5)
    columns: ['date', 'close', 'log_return', 'R_t_to_tk', 'y_L1']
    after date sort: shape=(6752, 2), same_len=True, same_dates=True

  Loading label 'L2' from: ..\data\derived\mru_usd_labels_L2_ma1

In [ ]:
# B1.4 – Define V1/V2 split masks based on validation_splits.md and inspect them


import pandas as pd

def ensure_datetime_date(df: pd.DataFrame, col: str = "date") -> pd.Series:
    """
    df[col] is datetime64[ns], converting in-place if necessary.
    Returns the datetime Series.
    """
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")
    return df[col]


def get_v1_masks(df: pd.DataFrame):
    """
    Build boolean masks for V1-train, V1-val, V1-test using validation_splits.md boundaries.

    Returns
    -------
    masks : dict
        {
          "train": mask_train,
          "val": mask_val,
          "test": mask_test,
        }
    """
    dates = ensure_datetime_date(df, "date")

    
    train_start = pd.Timestamp("2000-01-03")
    train_end   = pd.Timestamp("2015-12-31")
    val_start   = pd.Timestamp("2016-01-01")
    val_end     = pd.Timestamp("2019-12-31")
    test_start  = pd.Timestamp("2020-01-01")
    test_end    = pd.Timestamp("2025-11-25")

    mask_train = (dates >= train_start) & (dates <= train_end)
    mask_val   = (dates >= val_start) & (dates <= val_end)
    mask_test  = (dates >= test_start) & (dates <= test_end)

    print("=== V1 masks summary ===")
    for name, m in [("train", mask_train), ("val", mask_val), ("test", mask_test)]:
        n = int(m.sum())
        if n > 0:
            dmin = dates[m].min()
            dmax = dates[m].max()
        else:
            dmin = dmax = None
        print(f"V1-{name:5s}: count={n:4d}, date min/max = {dmin} -> {dmax}")

    # Overlap checks
    print("\nOverlap checks (should all be 0):")
    print("  train ∩ val  :", int((mask_train & mask_val).sum()))
    print("  train ∩ test :", int((mask_train & mask_test).sum()))
    print("  val   ∩ test :", int((mask_val & mask_test).sum()))

    total_in_any = int((mask_train | mask_val | mask_test).sum())
    print(f"\nTotal covered by V1 masks: {total_in_any} (df rows: {len(df)})")

    return {"train": mask_train, "val": mask_val, "test": mask_test}


def get_v2_folds(df: pd.DataFrame):
    """
    Build boolean masks for V2 walk-forward folds F1–F4 using  boundaries.

    Returns
    -------
    folds : dict
        {
          "F1": {"train": mask_train_F1, "test": mask_test_F1},
          "F2": {"train": mask_train_F2, "test": mask_test_F2},
          "F3": {"train": mask_train_F3, "test": mask_test_F3},
          "F4": {"train": mask_train_F4, "test": mask_test_F4},
        }
    """
    dates = ensure_datetime_date(df, "date")

    fold_defs = {
        "F1": {
            "train_start": "2000-01-03",
            "train_end":   "2009-12-31",
            "test_start":  "2010-01-01",  
            "test_end":    "2011-12-31",
        },
        "F2": {
            "train_start": "2000-01-03",
            "train_end":   "2011-12-31",
            "test_start":  "2012-01-01",
            "test_end":    "2013-12-31",
        },
        "F3": {
            "train_start": "2000-01-03",
            "train_end":   "2013-12-31",
            "test_start":  "2014-01-01",
            "test_end":    "2015-12-31",
        },
        "F4": {
            "train_start": "2000-01-03",
            "train_end":   "2015-12-31",
            "test_start":  "2016-01-01",
            "test_end":    "2019-12-31",
        },
    }

    folds = {}
    print("\n=== V2 folds summary ===")
    for fold_name, spec in fold_defs.items():
        ts = pd.Timestamp(spec["train_start"])
        te = pd.Timestamp(spec["train_end"])
        vs = pd.Timestamp(spec["test_start"])
        ve = pd.Timestamp(spec["test_end"])

        m_train = (dates >= ts) & (dates <= te)
        m_test  = (dates >= vs) & (dates <= ve)

        n_train = int(m_train.sum())
        n_test  = int(m_test.sum())
        dmin_train = dates[m_train].min() if n_train > 0 else None
        dmax_train = dates[m_train].max() if n_train > 0 else None
        dmin_test  = dates[m_test].min() if n_test > 0 else None
        dmax_test  = dates[m_test].max() if n_test > 0 else None

        print(f"{fold_name}:")
        print(f"  train: count={n_train:4d}, {dmin_train} -> {dmax_train}")
        print(f"  test : count={n_test:4d}, {dmin_test} -> {dmax_test}")
        print(f"  overlap train∩test: {int((m_train & m_test).sum())}")

        folds[fold_name] = {"train": m_train, "test": m_test}

    return folds



v1_masks = get_v1_masks(modeling_df)
v2_folds = get_v2_folds(modeling_df)



=== V1 masks summary ===
V1-train: count=4169, date min/max = 2000-01-03 00:00:00 -> 2015-12-31 00:00:00
V1-val  : count=1043, date min/max = 2016-01-01 00:00:00 -> 2019-12-31 00:00:00
V1-test : count=1540, date min/max = 2020-01-01 00:00:00 -> 2025-11-25 00:00:00

Overlap checks (should all be 0):
  train ∩ val  : 0
  train ∩ test : 0
  val   ∩ test : 0

Total covered by V1 masks: 6752 (df rows: 6752)

=== V2 folds summary ===
F1:
  train: count=2604, 2000-01-03 00:00:00 -> 2009-12-31 00:00:00
  test : count= 521, 2010-01-01 00:00:00 -> 2011-12-30 00:00:00
  overlap train∩test: 0
F2:
  train: count=3125, 2000-01-03 00:00:00 -> 2011-12-30 00:00:00
  test : count= 522, 2012-01-02 00:00:00 -> 2013-12-31 00:00:00
  overlap train∩test: 0
F3:
  train: count=3647, 2000-01-03 00:00:00 -> 2013-12-31 00:00:00
  test : count= 522, 2014-01-01 00:00:00 -> 2015-12-31 00:00:00
  overlap train∩test: 0
F4:
  train: count=4169, 2000-01-03 00:00:00 -> 2015-12-31 00:00:00
  test : count=1043, 2016-01-01 

In [ ]:
# B2.1 – Extract clean X/y for V1 (L1_Q + F_CORE) with diagnostics

import numpy as np
import pandas as pd
from collections import Counter

def get_feature_columns(config: dict, feature_set_key: str = "F_CORE"):
    """
    Return list of feature column names for a given feature set key
    (e.g., 'F_CORE', 'F_MIN')
    """
    feats_cfg = config["features"]
    if feature_set_key not in feats_cfg:
        raise KeyError(f"Feature set '{feature_set_key}' not found in config['features']. "
                       f"Available: {list(feats_cfg.keys())}")
    cols = feats_cfg[feature_set_key]["columns"]
    print(f"[get_feature_columns] feature_set='{feature_set_key}', columns={cols}")
    return cols


def get_label_column(config: dict, label_key: str = "L1_Q") -> str:
    """
    Return label column name for a given label key (e.g., 'L1_Q', 'L2').
    """
    labels_cfg = config["labels"]
    if label_key not in labels_cfg:
        raise KeyError(f"Label '{label_key}' not found in config['labels']. "
                       f"Available: {list(labels_cfg.keys())}")
    lbl_col = labels_cfg[label_key]["label_column"]
    print(f"[get_label_column] label_key='{label_key}', label_column='{lbl_col}'")
    return lbl_col


def extract_xy_from_mask(
    df: pd.DataFrame,
    config: dict,
    label_key: str = "L1_Q",
    feature_set_key: str = "F_CORE",
    mask: pd.Series = None,
    dropna: bool = True,
):
    """
    Extract (X, y) for a given label and feature set, restricted to rows where mask==True.

    - Uses config to determine feature and label columns.
    -  drops any rows where label or any feature is NaN.
    - Prints diagnostics: sizes, date range, NaN counts, class distribution.

    Returns
    -------
    X : np.ndarray of shape (n_samples, n_features)
    y : np.ndarray of shape (n_samples,)
    subset_df : pd.DataFrame (the filtered subset, with date + features + label)
    """
    if mask is not None:
        subset = df.loc[mask].copy()
    else:
        subset = df.copy()

    dates = subset["date"]
    feat_cols = get_feature_columns(config, feature_set_key=feature_set_key)
    label_col = get_label_column(config, label_key=label_key)

    print("\n" + "=" * 80)
    print(f"[extract_xy_from_mask] label='{label_key}', feature_set='{feature_set_key}'")
    print(f"Initial subset shape (before NaN filtering): {subset.shape}")
    print(f"Date range in subset: {dates.min()} -> {dates.max()}")

    missing_feats = [c for c in feat_cols if c not in subset.columns]
    if missing_feats:
        print(f"WARNING: missing feature columns in subset: {missing_feats}")
    if label_col not in subset.columns:
        print(f"WARNING: label column '{label_col}' not found in subset columns!")

    cols_to_check = feat_cols + [label_col]
    nan_counts = subset[cols_to_check].isna().sum()
    print("\nNaN counts before dropna (features + label):")
    print(nan_counts)

    if dropna:
        valid_mask = subset[cols_to_check].notna().all(axis=1)
        n_total = len(subset)
        n_valid = int(valid_mask.sum())
        n_dropped = n_total - n_valid
        print(f"\nDropping rows with any NaN in features or label.")
        print(f"Total rows: {n_total}, valid rows: {n_valid}, dropped: {n_dropped}")

        subset = subset.loc[valid_mask].copy()
        dates = subset["date"]

    # Build X, y
    X = subset[feat_cols].to_numpy(dtype=float)
    y = subset[label_col].to_numpy()

    print(f"\nFinal subset shape after NaN filtering: {subset.shape}")
    print(f"X shape: {X.shape}, y shape: {y.shape}")
    if len(subset) > 0:
        print(f"Final date range: {dates.min()} -> {dates.max()}")

    # Label distribution
    label_counts = Counter(y)
    print("\nLabel value counts in this subset (after NaN filtering):")
    for k in sorted(label_counts.keys(), key=lambda x: (-999 if np.isnan(x) else x)):
        print(f"  {k}: {label_counts[k]}")

    return X, y, subset



print("\n=== B2.1 – V1: L1_Q + F_CORE extraction diagnostics ===")

X_train, y_train, df_train = extract_xy_from_mask(
    modeling_df, config, label_key="L1_Q", feature_set_key="F_CORE", mask=v1_masks["train"], dropna=True
)

X_val, y_val, df_val = extract_xy_from_mask(
    modeling_df, config, label_key="L1_Q", feature_set_key="F_CORE", mask=v1_masks["val"], dropna=True
)

X_test, y_test, df_test = extract_xy_from_mask(
    modeling_df, config, label_key="L1_Q", feature_set_key="F_CORE", mask=v1_masks["test"], dropna=True
)

print("\n=== Summary of V1 shapes (after NaN filtering) ===")
print(f"train: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"val  : X_val  ={X_val.shape},   y_val  ={y_val.shape}")
print(f"test : X_test ={X_test.shape},  y_test ={y_test.shape}")




=== B2.1 – V1: L1_Q + F_CORE extraction diagnostics ===
[get_feature_columns] feature_set='F_CORE', columns=['f_r_lag1', 'f_past_ret_5', 'f_past_ret_20', 'f_vol_10', 'f_vol_30']
[get_label_column] label_key='L1_Q', label_column='y_L1_Q'

[extract_xy_from_mask] label='L1_Q', feature_set='F_CORE'
Initial subset shape (before NaN filtering): (4169, 22)
Date range in subset: 2000-01-03 00:00:00 -> 2015-12-31 00:00:00

NaN counts before dropna (features + label):
f_r_lag1          2
f_past_ret_5      5
f_past_ret_20    20
f_vol_10         11
f_vol_30         31
y_L1_Q            0
dtype: int64

Dropping rows with any NaN in features or label.
Total rows: 4169, valid rows: 4138, dropped: 31

Final subset shape after NaN filtering: (4138, 22)
X shape: (4138, 5), y shape: (4138,)
Final date range: 2000-02-15 00:00:00 -> 2015-12-31 00:00:00

Label value counts in this subset (after NaN filtering):
  -1.0: 834
  0.0: 2475
  1.0: 829
[get_feature_columns] feature_set='F_CORE', columns=['f_r_lag1

In [ ]:
# V1-aware Logistic Regression for L1_Q + F_CORE (E1 core)

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


def fit_logistic_v1_L1Q_FCORE(
    df,
    config,
    v1_masks,
    C: float = 1.0,
    penalty: str = "l2",
    solver: str = "lbfgs",
    max_iter: int = 1000,
):
    """
    Fit a multinomial Logistic Regression classifier on V1-train for
    label=L1_Q and features=F_CORE, then evaluate on train/val/test.

    This is the concrete implementation of experiment E1 under V1.

    Returns
    -------
    results : dict
        {
          "model": fitted LogisticRegression instance,
          "scaler": fitted StandardScaler,
          "metrics": {
            "train": {...},
            "val": {...},
            "test": {...},
          }
        }
    """
    print("=== B2.2 – Extracting V1 splits for L1_Q + F_CORE ===")
    X_train, y_train, df_train = extract_xy_from_mask(
        df, config,
        label_key="L1_Q",
        feature_set_key="F_CORE",
        mask=v1_masks["train"],
        dropna=True,
    )
    X_val, y_val, df_val = extract_xy_from_mask(
        df, config,
        label_key="L1_Q",
        feature_set_key="F_CORE",
        mask=v1_masks["val"],
        dropna=True,
    )
    X_test, y_test, df_test = extract_xy_from_mask(
        df, config,
        label_key="L1_Q",
        feature_set_key="F_CORE",
        mask=v1_masks["test"],
        dropna=True,
    )

    print("\n=== B2.2 – Fitting StandardScaler on V1-train only ===")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    print("Scaler fitted. Feature-wise mean (train):")
    print(scaler.mean_)
    print("Feature-wise scale (train):")
    print(scaler.scale_)

    print("\n=== B2.2 – Fitting Logistic Regression (E1) on V1-train ===")
    log_reg = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        multi_class="auto",
        max_iter=max_iter,
    )
    log_reg.fit(X_train_scaled, y_train)

    print("Model fitted.")
    print("Coefficients shape:", log_reg.coef_.shape)
    print("Intercepts shape:", log_reg.intercept_.shape)
    classes = log_reg.classes_
    print("Model classes:", classes)

    def evaluate_split(name, X_split, y_split):
        y_pred = log_reg.predict(X_split)

        acc = accuracy_score(y_split, y_pred)
        macro_f1 = f1_score(y_split, y_pred, average="macro")

        print("\n" + "=" * 80)
        print(f"[E1 – Logistic, V1] Evaluation on {name}")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Macro F1:  {macro_f1:.4f}")
        print("\nClassification report:")
        print(classification_report(y_split, y_pred, digits=4))

        cm = confusion_matrix(y_split, y_pred, labels=classes)
        print("Confusion matrix (rows=true, cols=pred, class order={}):".format(list(classes)))
        print(cm)

        metrics_dict = {
            "accuracy": acc,
            "macro_f1": macro_f1,
            "confusion_matrix": cm,
            "classes": classes,
        }
        return metrics_dict

    metrics_train = evaluate_split("V1-train", X_train_scaled, y_train)
    metrics_val = evaluate_split("V1-val", X_val_scaled, y_val)
    metrics_test = evaluate_split("V1-test", X_test_scaled, y_test)

    results = {
        "model": log_reg,
        "scaler": scaler,
        "metrics": {
            "train": metrics_train,
            "val": metrics_val,
            "test": metrics_test,
        },
    }

    print("\n=== B2.2 – Summary of E1 (Logistic, V1) ===")
    print(f"Train accuracy: {metrics_train['accuracy']:.4f}, macro F1: {metrics_train['macro_f1']:.4f}")
    print(f"Val   accuracy: {metrics_val['accuracy']:.4f}, macro F1: {metrics_val['macro_f1']:.4f}")
    print(f"Test  accuracy: {metrics_test['accuracy']:.4f}, macro F1: {metrics_test['macro_f1']:.4f}")

    return results


e1_results = fit_logistic_v1_L1Q_FCORE(modeling_df, config, v1_masks)



=== B2.2 – Extracting V1 splits for L1_Q + F_CORE ===
[get_feature_columns] feature_set='F_CORE', columns=['f_r_lag1', 'f_past_ret_5', 'f_past_ret_20', 'f_vol_10', 'f_vol_30']
[get_label_column] label_key='L1_Q', label_column='y_L1_Q'

[extract_xy_from_mask] label='L1_Q', feature_set='F_CORE'
Initial subset shape (before NaN filtering): (4169, 22)
Date range in subset: 2000-01-03 00:00:00 -> 2015-12-31 00:00:00

NaN counts before dropna (features + label):
f_r_lag1          2
f_past_ret_5      5
f_past_ret_20    20
f_vol_10         11
f_vol_30         31
y_L1_Q            0
dtype: int64

Dropping rows with any NaN in features or label.
Total rows: 4169, valid rows: 4138, dropped: 31

Final subset shape after NaN filtering: (4138, 22)
X shape: (4138, 5), y shape: (4138,)
Final date range: 2000-02-15 00:00:00 -> 2015-12-31 00:00:00

Label value counts in this subset (after NaN filtering):
  -1.0: 834
  0.0: 2475
  1.0: 829
[get_feature_columns] feature_set='F_CORE', columns=['f_r_lag1', 

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [ ]:
# B2.3 – V1-aware Linear SVM for L1_Q + F_CORE (E3 core)

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


def fit_linear_svm_v1_L1Q_FCORE(
    df,
    config,
    v1_masks,
    C: float = 1.0,
    class_weight=None,      # we wll try "balanced"
    max_iter: int = 5000,
):
    """
    Fit a linear SVM (LinearSVC) on V1-train for label=L1_Q, features=F_CORE,
    then evaluate on V1-train, V1-val, V1-test.

    This is experiment E3 under V1.
    """

    # --- 1. Extract X/y for each split (same helper as in E1, no leakage) ---
    print("=== B2.3 – Extracting V1 splits for L1_Q + F_CORE (SVM) ===")
    X_train, y_train, df_train = extract_xy_from_mask(
        df, config,
        label_key="L1_Q",
        feature_set_key="F_CORE",
        mask=v1_masks["train"],
        dropna=True,
    )
    X_val, y_val, df_val = extract_xy_from_mask(
        df, config,
        label_key="L1_Q",
        feature_set_key="F_CORE",
        mask=v1_masks["val"],
        dropna=True,
    )
    X_test, y_test, df_test = extract_xy_from_mask(
        df, config,
        label_key="L1_Q",
        feature_set_key="F_CORE",
        mask=v1_masks["test"],
        dropna=True,
    )

    # --- 2. Fit StandardScaler on V1-train only ---
    print("\n=== B2.3 – Fitting StandardScaler on V1-train only (SVM) ===")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    print("Scaler fitted. Feature-wise mean (train):")
    print(scaler.mean_)
    print("Feature-wise scale (train):")
    print(scaler.scale_)

    # --- 3. Fit LinearSVC on V1-train ---
    print("\n=== B2.3 – Fitting LinearSVC (E3) on V1-train ===")
    svm = LinearSVC(
        C=C,
        class_weight=class_weight,
        max_iter=max_iter,
    )
    svm.fit(X_train_scaled, y_train)

    print("Model fitted.")
    print("Classes seen during training:", svm.classes_)

    # --- 4. Evaluation helper ---
    def evaluate_split(name, X_split, y_split):
        y_pred = svm.predict(X_split)

        acc = accuracy_score(y_split, y_pred)
        macro_f1 = f1_score(y_split, y_pred, average="macro")

        print("\n" + "=" * 80)
        print(f"[E3 – Linear SVM, V1] Evaluation on {name}")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Macro F1:  {macro_f1:.4f}")
        print("\nClassification report:")
        print(classification_report(y_split, y_pred, digits=4))

        classes = svm.classes_
        cm = confusion_matrix(y_split, y_pred, labels=classes)
        print("Confusion matrix (rows=true, cols=pred, class order={}):".format(list(classes)))
        print(cm)

        return {
            "accuracy": acc,
            "macro_f1": macro_f1,
            "confusion_matrix": cm,
            "classes": classes,
        }

    # --- 5. Evaluate on train, val, test ---
    metrics_train = evaluate_split("V1-train", X_train_scaled, y_train)
    metrics_val = evaluate_split("V1-val", X_val_scaled, y_val)
    metrics_test = evaluate_split("V1-test", X_test_scaled, y_test)

    print("\n=== B2.3 – Summary of E3 (Linear SVM, V1) ===")
    print(f"Train accuracy: {metrics_train['accuracy']:.4f}, macro F1: {metrics_train['macro_f1']:.4f}")
    print(f"Val   accuracy: {metrics_val['accuracy']:.4f}, macro F1: {metrics_val['macro_f1']:.4f}")
    print(f"Test  accuracy: {metrics_test['accuracy']:.4f}, macro F1: {metrics_test['macro_f1']:.4f}")

    results = {
        "model": svm,
        "scaler": scaler,
        "metrics": {
            "train": metrics_train,
            "val": metrics_val,
            "test": metrics_test,
        },
    }
    return results



e3_results = fit_linear_svm_v1_L1Q_FCORE(modeling_df, config, v1_masks)



=== B2.3 – Extracting V1 splits for L1_Q + F_CORE (SVM) ===
[get_feature_columns] feature_set='F_CORE', columns=['f_r_lag1', 'f_past_ret_5', 'f_past_ret_20', 'f_vol_10', 'f_vol_30']
[get_label_column] label_key='L1_Q', label_column='y_L1_Q'

[extract_xy_from_mask] label='L1_Q', feature_set='F_CORE'
Initial subset shape (before NaN filtering): (4169, 22)
Date range in subset: 2000-01-03 00:00:00 -> 2015-12-31 00:00:00

NaN counts before dropna (features + label):
f_r_lag1          2
f_past_ret_5      5
f_past_ret_20    20
f_vol_10         11
f_vol_30         31
y_L1_Q            0
dtype: int64

Dropping rows with any NaN in features or label.
Total rows: 4169, valid rows: 4138, dropped: 31

Final subset shape after NaN filtering: (4138, 22)
X shape: (4138, 5), y shape: (4138,)
Final date range: 2000-02-15 00:00:00 -> 2015-12-31 00:00:00

Label value counts in this subset (after NaN filtering):
  -1.0: 834
  0.0: 2475
  1.0: 829
[get_feature_columns] feature_set='F_CORE', columns=['f_r_l

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [ ]:
# B3 – Metrics tables for E1 (Logistic) and E3 (Linear SVM), V1, L1_Q + F_CORE

import pandas as pd
from pathlib import Path


ROOT = Path("..")
RESULTS_TABLES = ROOT / "results" / "tables"
RESULTS_TABLES.mkdir(parents=True, exist_ok=True)


def metrics_dict_to_df(exp_id: str, metrics: dict) -> pd.DataFrame:
    """
    Convert the 'metrics' dict from e1_results/e3_results into a tidy DataFrame
    with one row per split (train/val/test) and columns:
        experiment, split, accuracy, macro_f1
    """
    rows = []
    for split_name, m in metrics.items():
        rows.append(
            {
                "experiment": exp_id,
                "split": split_name,
                "accuracy": m["accuracy"],
                "macro_f1": m["macro_f1"],
            }
        )
    df = pd.DataFrame(rows)
    return df


# E1 – Logistic Regression, V1, L1_Q + F_CORE
print("=== B3 – Building metrics table for E1 (Logistic, V1, L1_Q, F_CORE) ===")
df_metrics_E1 = metrics_dict_to_df(
    exp_id="E1_logistic_V1_L1Q_FCORE",
    metrics=e1_results["metrics"],
)
print("\nE1 metrics table:")
print(df_metrics_E1)

e1_out_path = RESULTS_TABLES / "metrics_E1.csv"
df_metrics_E1.to_csv(e1_out_path, index=False)
print(f"\nSaved E1 metrics to: {e1_out_path}")

# E3 – Linear SVM, V1, L1_Q + F_CORE

print("\n=== B3 – Building metrics table for E3 (Linear SVM, V1, L1_Q, F_CORE) ===")
df_metrics_E3 = metrics_dict_to_df(
    exp_id="E3_linearSVM_V1_L1Q_FCORE",
    metrics=e3_results["metrics"],
)
print("\nE3 metrics table:")
print(df_metrics_E3)

e3_out_path = RESULTS_TABLES / "metrics_E3.csv"
df_metrics_E3.to_csv(e3_out_path, index=False)
print(f"\nSaved E3 metrics to: {e3_out_path}")



=== B3 – Building metrics table for E1 (Logistic, V1, L1_Q, F_CORE) ===

E1 metrics table:
                 experiment  split  accuracy  macro_f1
0  E1_logistic_V1_L1Q_FCORE  train  0.592798  0.260366
1  E1_logistic_V1_L1Q_FCORE    val  0.786194  0.293434
2  E1_logistic_V1_L1Q_FCORE   test  0.731596  0.281665

Saved E1 metrics to: ..\results\tables\metrics_E1.csv

=== B3 – Building metrics table for E3 (Linear SVM, V1, L1_Q, F_CORE) ===

E3 metrics table:
                  experiment  split  accuracy  macro_f1
0  E3_linearSVM_V1_L1Q_FCORE  train  0.593040  0.260326
1  E3_linearSVM_V1_L1Q_FCORE    val  0.786194  0.293434
2  E3_linearSVM_V1_L1Q_FCORE   test  0.731596  0.281665

Saved E3 metrics to: ..\results\tables\metrics_E3.csv

=== End of B3 – metrics tables saved for E1 and E3 ===
Please copy ALL printed output from this cell back into ChatGPT.


In [ ]:
# V2 walk-forward experiments (E2 & E4) for L1_Q + F_CORE

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

# built exactly as in B1/B2:
#   - one row per date
#   - columns: features + y_L1_Q + y_L1 + y_L2 + MA_* etc.


# 2.1 – Define V2 folds (date ranges) and build masks from modeling_df["date"]

def make_v2_folds_from_dates(df: pd.DataFrame):
    """
    Reconstruct V2 folds F1–F4 using the date ranges that were already
    used in B1.4 .

    Returns:
        folds: dict of
          fold_name -> {
              "train_mask": boolean array,
              "test_mask":  boolean array,
          }
    """
    d = df["date"]

    # NOTE: Boundaries chosen to match the B1.4 .
    # Train ranges:
    #   F1: 2000-01-03 -> 2009-12-31
    #   F2: 2000-01-03 -> 2011-12-30
    #   F3: 2000-01-03 -> 2013-12-31
    #   F4: 2000-01-03 -> 2015-12-31
    #
    # Test ranges:
    #   F1: 2010-01-01 -> 2011-12-30
    #   F2: 2012-01-02 -> 2013-12-31
    #   F3: 2014-01-01 -> 2015-12-31
    #   F4: 2016-01-01 -> 2019-12-31

    boundaries = {
        "F1": {
            "train_start": datetime(2000, 1, 3),
            "train_end":   datetime(2009, 12, 31),
            "test_start":  datetime(2010, 1, 1),
            "test_end":    datetime(2011, 12, 30),
        },
        "F2": {
            "train_start": datetime(2000, 1, 3),
            "train_end":   datetime(2011, 12, 30),
            "test_start":  datetime(2012, 1, 2),
            "test_end":    datetime(2013, 12, 31),
        },
        "F3": {
            "train_start": datetime(2000, 1, 3),
            "train_end":   datetime(2013, 12, 31),
            "test_start":  datetime(2014, 1, 1),
            "test_end":    datetime(2015, 12, 31),
        },
        "F4": {
            "train_start": datetime(2000, 1, 3),
            "train_end":   datetime(2015, 12, 31),
            "test_start":  datetime(2016, 1, 1),
            "test_end":    datetime(2019, 12, 31),
        },
    }

    folds = {}
    print("=== V2 folds (reconstructed from dates) ===")
    for fold_name, b in boundaries.items():
        train_mask = (d >= b["train_start"]) & (d <= b["train_end"])
        test_mask  = (d >= b["test_start"]) & (d <= b["test_end"])

        folds[fold_name] = {
            "train_mask": train_mask,
            "test_mask":  test_mask,
        }

        print(f"{fold_name}:")
        print(
            f"  train: count={train_mask.sum():4d}, "
            f"{d[train_mask].min()} -> {d[train_mask].max()}"
        )
        print(
            f"  test : count={test_mask.sum():4d}, "
            f"{d[test_mask].min()} -> {d[test_mask].max()}"
        )
        overlap = (train_mask & test_mask).sum()
        print(f"  overlap train∩test: {overlap}")
    print("=== End V2 folds reconstruction ===\n")
    return folds


# 2.2 – Helper to extract X, y for a fold (using date-based masks)

def extract_xy_for_fold(df: pd.DataFrame,
                        train_mask: pd.Series,
                        test_mask: pd.Series,
                        feature_cols,
                        label_col: str):


    subset_train = df.loc[train_mask, feature_cols + [label_col]].copy()
    subset_test  = df.loc[test_mask,  feature_cols + [label_col]].copy()

    print(f"[extract_xy_for_fold] label='{label_col}', "
          f"n_train_raw={subset_train.shape[0]}, n_test_raw={subset_test.shape[0]}")

    print("\nNaN counts in TRAIN (features + label):")
    print(subset_train.isna().sum())

    print("\nNaN counts in TEST (features + label):")
    print(subset_test.isna().sum())

    subset_train = subset_train.dropna()
    subset_test  = subset_test.dropna()

    X_train = subset_train[feature_cols].values
    y_train = subset_train[label_col].values
    X_test  = subset_test[feature_cols].values
    y_test  = subset_test[label_col].values

    print(
        f"\nAfter dropna: "
        f"n_train={X_train.shape[0]}, n_test={X_test.shape[0]}"
    )

    # Label distributions
    print("\nLabel value counts (TRAIN):")
    print(pd.Series(y_train).value_counts().sort_index())
    print("\nLabel value counts (TEST):")
    print(pd.Series(y_test).value_counts().sort_index())

    return X_train, y_train, X_test, y_test


# 2.3 – Generic V2 runner for linear models (Logistic / Linear SVM)

def run_linear_experiment_V2(
    exp_id: str,
    df: pd.DataFrame,
    folds: dict,
    feature_cols,
    label_col: str,
    model_type: str = "logistic",
    C: float = 1.0,
):
    """
    Run a linear model (Logistic or LinearSVC) in a walk-forward fashion over V2 folds.

    For each fold:
        - extract X_train, y_train, X_test, y_test
        - fit StandardScaler on train only
        - train model
        - evaluate on train and test
        - print reports and confusion matrices
        - collect metrics into a list
    Returns:
        metrics_df: DataFrame with columns [experiment, fold, split, accuracy, macro_f1]
    """
    assert model_type in {"logistic", "linear_svm"}

    print(f"=== {exp_id} – V2 walk-forward ({model_type}) ===")
    all_metrics = []

    for fold_name, masks in folds.items():
        print("\n" + "=" * 80)
        print(f"Fold {fold_name} – extracting data")
        train_mask = masks["train_mask"]
        test_mask  = masks["test_mask"]

        X_train, y_train, X_test, y_test = extract_xy_for_fold(
            df, train_mask, test_mask, feature_cols, label_col
        )

        # Scale features (fit on TRAIN only)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        # Choose model
        if model_type == "logistic":
            model = LogisticRegression(
                C=C,
                penalty="l2",
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=200,
                class_weight=None,
            )
        else:  # linear_svm
            model = LinearSVC(
                C=C,
                class_weight=None,
                max_iter=2000,
            )

        print(f"\nTraining {model_type} on fold {fold_name}...")
        model.fit(X_train_scaled, y_train)

        # Evaluate on TRAIN
        y_pred_train = model.predict(X_train_scaled)
        acc_train = accuracy_score(y_train, y_pred_train)
        macro_f1_train = f1_score(y_train, y_pred_train, average="macro")

        print(f"\n[{exp_id} – {fold_name}] TRAIN metrics")
        print(f"Accuracy:  {acc_train:.4f}")
        print(f"Macro F1:  {macro_f1_train:.4f}")
        print("\nClassification report (TRAIN):")
        print(classification_report(y_train, y_pred_train))
        print("Confusion matrix (TRAIN):")
        print(confusion_matrix(y_train, y_pred_train))

        all_metrics.append({
            "experiment": exp_id,
            "fold": fold_name,
            "split": "train",
            "accuracy": acc_train,
            "macro_f1": macro_f1_train,
        })

        # Evaluate on TEST
        y_pred_test = model.predict(X_test_scaled)
        acc_test = accuracy_score(y_test, y_pred_test)
        macro_f1_test = f1_score(y_test, y_pred_test, average="macro")

        print(f"\n[{exp_id} – {fold_name}] TEST metrics")
        print(f"Accuracy:  {acc_test:.4f}")
        print(f"Macro F1:  {macro_f1_test:.4f}")
        print("\nClassification report (TEST):")
        print(classification_report(y_test, y_pred_test))
        print("Confusion matrix (TEST):")
        print(confusion_matrix(y_test, y_pred_test))

        all_metrics.append({
            "experiment": exp_id,
            "fold": fold_name,
            "split": "test",
            "accuracy": acc_test,
            "macro_f1": macro_f1_test,
        })

    metrics_df = pd.DataFrame(all_metrics)
    print("\n=== Aggregated metrics for", exp_id, "===")
    print(metrics_df)
    return metrics_df


# 2.4 – Run E2 (Logistic, V2) and E4 (Linear SVM, V2) with L1_Q + F_CORE

# Define F_CORE explicitly (to avoid any ambiguity)
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]
label_col = "y_L1_Q"

# Rebuild V2 folds from modeling_df
v2_folds = make_v2_folds_from_dates(modeling_df)

# E2 – Logistic Regression, V2
metrics_E2 = run_linear_experiment_V2(
    exp_id="E2_logistic_V2_L1Q_FCORE",
    df=modeling_df,
    folds=v2_folds,
    feature_cols=F_CORE,
    label_col=label_col,
    model_type="logistic",
    C=1.0,
)

# E4 – Linear SVM, V2
metrics_E4 = run_linear_experiment_V2(
    exp_id="E4_linearSVM_V2_L1Q_FCORE",
    df=modeling_df,
    folds=v2_folds,
    feature_cols=F_CORE,
    label_col=label_col,
    model_type="linear_svm",
    C=1.0,
)

# Save metrics to CSV
ROOT = Path("..")
tables_dir = ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

path_E2 = tables_dir / "metrics_E2.csv"
path_E4 = tables_dir / "metrics_E4.csv"

metrics_E2.to_csv(path_E2, index=False)
metrics_E4.to_csv(path_E4, index=False)

print(f"\nSaved E2 metrics to: {path_E2}")
print(f"Saved E4 metrics to: {path_E4}")
print("\n=== End of Step 2 (E2 & E4 V2 runs) ===")


=== V2 folds (reconstructed from dates) ===
F1:
  train: count=2604, 2000-01-03 00:00:00 -> 2009-12-31 00:00:00
  test : count= 521, 2010-01-01 00:00:00 -> 2011-12-30 00:00:00
  overlap train∩test: 0
F2:
  train: count=3125, 2000-01-03 00:00:00 -> 2011-12-30 00:00:00
  test : count= 522, 2012-01-02 00:00:00 -> 2013-12-31 00:00:00
  overlap train∩test: 0
F3:
  train: count=3647, 2000-01-03 00:00:00 -> 2013-12-31 00:00:00
  test : count= 522, 2014-01-01 00:00:00 -> 2015-12-31 00:00:00
  overlap train∩test: 0
F4:
  train: count=4169, 2000-01-03 00:00:00 -> 2015-12-31 00:00:00
  test : count=1043, 2016-01-01 00:00:00 -> 2019-12-31 00:00:00
  overlap train∩test: 0
=== End V2 folds reconstruction ===

=== E2_logistic_V2_L1Q_FCORE – V2 walk-forward (logistic) ===

Fold F1 – extracting data
--------------------------------------------------
[extract_xy_for_fold] label='y_L1_Q', n_train_raw=2604, n_test_raw=521

NaN counts in TRAIN (features + label):
f_r_lag1          2
f_past_ret_5      5
f_p

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

-1.0    149
 0.0    247
 1.0    126
Name: count, dtype: int64

Training logistic on fold F3...

[E2_logistic_V2_L1Q_FCORE – F3] TRAIN metrics
Accuracy:  0.6123
Macro F1:  0.2632

Classification report (TRAIN):
              precision    recall  f1-score   support

        -1.0       0.11      0.00      0.01       685
         0.0       0.62      0.99      0.76      2228
         1.0       0.45      0.01      0.02       703

    accuracy                           0.61      3616
   macro avg       0.39      0.33      0.26      3616
weighted avg       0.49      0.61      0.47      3616

Confusion matrix (TRAIN):
[[   2  682    1]
 [  15 2203   10]
 [   2  692    9]]

[E2_logistic_V2_L1Q_FCORE – F3] TEST metrics
Accuracy:  0.4655
Macro F1:  0.2167

Classification report (TEST):
              precision    recall  f1-score   support

        -1.0       0.00      0.00      0.00       149
         0.0       0.47      0.98      0.64       247
         1.0       0.14      0.01      0.02       12

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [ ]:
# Aggregate and compare linear baselines (E1, E3, E2, E4)

import pandas as pd
from pathlib import Path
import numpy as np

import plotly.express as px  

ROOT = Path("..")
tables_dir = ROOT / "results" / "tables"

paths = {
    "E1": tables_dir / "metrics_E1.csv",  # Logistic, V1
    "E3": tables_dir / "metrics_E3.csv",  # Linear SVM, V1
    "E2": tables_dir / "metrics_E2.csv",  # Logistic, V2
    "E4": tables_dir / "metrics_E4.csv",  # Linear SVM, V2
}

print("=== Step 3 – Loading metrics CSVs ===")
for k, p in paths.items():
    print(f"{k}: {p}  (exists={p.exists()})")

# Load and tag each table
def load_and_tag(exp_key, path):
    df = pd.read_csv(path)
    df = df.copy()
    df["exp_key"] = exp_key

    # Decode scheme and model from experiment name
    exp_name = df["experiment"].iloc[0]
    if "_V1_" in exp_name:
        scheme = "V1"
    elif "_V2_" in exp_name:
        scheme = "V2"
    else:
        scheme = "UNKNOWN"

    if "logistic" in exp_name.lower():
        model = "logistic"
    elif "linearSVM".lower() in exp_name.lower():
        model = "linear_svm"
    elif "linearSVM" in exp_name:
        model = "linear_svm"
    elif "linear" in exp_name.lower():
        model = "linear_svm"
    else:
        model = "UNKNOWN"

    df["scheme"] = scheme
    df["model"] = model
    df["label"] = "L1_Q"
    df["feature_set"] = "F_CORE"
    return df

metrics_list = []
for k, p in paths.items():
    dfk = load_and_tag(k, p)
    metrics_list.append(dfk)

metrics_all = pd.concat(metrics_list, ignore_index=True)

print("\n=== All metrics (raw, long form) ===")
print(metrics_all)

# 3.1 – Aggregate V2 folds: mean metrics by experiment + split

# Separate V1 and V2
metrics_V1 = metrics_all[metrics_all["scheme"] == "V1"].copy()
metrics_V2 = metrics_all[metrics_all["scheme"] == "V2"].copy()

print("\n=== V1 metrics (no folds) ===")
print(metrics_V1)

print("\n=== V2 metrics (per-fold) ===")
print(metrics_V2)

# Aggregate V2 across folds
metrics_V2_agg = (
    metrics_V2
    .groupby(["experiment", "exp_key", "scheme", "model", "label", "feature_set", "split"])
    [["accuracy", "macro_f1"]]
    .mean()
    .reset_index()
)

print("\n=== V2 metrics aggregated across folds ===")
print(metrics_V2_agg)

# 3.2 – Compact comparison for TEST splits only

def get_test_rows(df):
    return df[df["split"] == "test"].copy()

V1_test = get_test_rows(metrics_V1)
V2_test_agg = get_test_rows(metrics_V2_agg)

print("\n=== V1 TEST metrics (E1, E3) ===")
print(V1_test[["experiment", "model", "scheme", "accuracy", "macro_f1"]])

print("\n=== V2 TEST metrics (E2, E4 aggregated across folds) ===")
print(V2_test_agg[["experiment", "model", "scheme", "accuracy", "macro_f1"]])

# Build a unified comparison table for test-only
test_summary = pd.concat(
    [
        V1_test[["experiment", "model", "scheme", "accuracy", "macro_f1"]],
        V2_test_agg[["experiment", "model", "scheme", "accuracy", "macro_f1"]],
    ],
    ignore_index=True,
)

print("\n=== Step 3 – Unified TEST summary (linear models, L1_Q + F_CORE) ===")
print(test_summary)


# Create a long-form table for plotting
plot_df = test_summary.copy()
plot_df_long = plot_df.melt(
    id_vars=["experiment", "model", "scheme"],
    value_vars=["accuracy", "macro_f1"],
    var_name="metric",
    value_name="value",
)

print("\n=== Data used for Plotly bar chart ===")
print(plot_df_long)

# Plotly bar chart: each experiment × metric
fig = px.bar(
    plot_df_long,
    x="experiment",
    y="value",
    color="metric",
    barmode="group",
    facet_col="scheme",
    title="Linear baselines – TEST metrics (V1 vs V2, L1_Q + F_CORE)",
)
fig.show()

# Textual summary of the same info (so ChatGPT can 'see' it)
print("\n=== Textual summary of TEST metrics by experiment ===")
for _, row in test_summary.iterrows():
    print(
        f"Experiment={row['experiment']}, scheme={row['scheme']}, model={row['model']}: "
        f"accuracy={row['accuracy']:.4f}, macro_f1={row['macro_f1']:.4f}"
    )


=== Step 3 – Loading metrics CSVs ===
E1: ..\results\tables\metrics_E1.csv  (exists=True)
E3: ..\results\tables\metrics_E3.csv  (exists=True)
E2: ..\results\tables\metrics_E2.csv  (exists=True)
E4: ..\results\tables\metrics_E4.csv  (exists=True)

=== All metrics (raw, long form) ===
                   experiment  split  accuracy  macro_f1 exp_key scheme       model label feature_set fold
0    E1_logistic_V1_L1Q_FCORE  train  0.592798  0.260366      E1     V1    logistic  L1_Q      F_CORE  NaN
1    E1_logistic_V1_L1Q_FCORE    val  0.786194  0.293434      E1     V1    logistic  L1_Q      F_CORE  NaN
2    E1_logistic_V1_L1Q_FCORE   test  0.731596  0.281665      E1     V1    logistic  L1_Q      F_CORE  NaN
3   E3_linearSVM_V1_L1Q_FCORE  train  0.593040  0.260326      E3     V1  linear_svm  L1_Q      F_CORE  NaN
4   E3_linearSVM_V1_L1Q_FCORE    val  0.786194  0.293434      E3     V1  linear_svm  L1_Q      F_CORE  NaN
5   E3_linearSVM_V1_L1Q_FCORE   test  0.731596  0.281665      E3     V1  l


=== Textual summary of TEST metrics by experiment ===
Experiment=E1_logistic_V1_L1Q_FCORE, scheme=V1, model=logistic: accuracy=0.7316, macro_f1=0.2817
Experiment=E3_linearSVM_V1_L1Q_FCORE, scheme=V1, model=linear_svm: accuracy=0.7316, macro_f1=0.2817
Experiment=E2_logistic_V2_L1Q_FCORE, scheme=V2, model=logistic: accuracy=0.5613, macro_f1=0.2382
Experiment=E4_linearSVM_V2_L1Q_FCORE, scheme=V2, model=linear_svm: accuracy=0.5622, macro_f1=0.2384


In [ ]:

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

import yaml

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix



def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "config" / "labels_features.yaml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate project root containing config/labels_features.yaml")

ROOT = find_project_root(Path.cwd())
CFG_PATH = ROOT / "config" / "labels_features.yaml"

with open(CFG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

OUT_DIR = ROOT / "results" / "figures" / "week6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("CFG :", CFG_PATH)
print("OUT :", OUT_DIR)



def load_features_labels(cfg: dict, label_keys: list[str]) -> pd.DataFrame:
    dataset = cfg["dataset"]
    labels_cfg = cfg["labels"]

    feat_path = ROOT / Path(dataset["features_file"])
    df = pd.read_csv(feat_path)

    if "date" not in df.columns:
        raise ValueError("Features file must contain a 'date' column.")
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
    df = df.sort_values("date").reset_index(drop=True)

    for lbl_key in label_keys:
        if lbl_key not in labels_cfg:
            raise KeyError(f"Label key '{lbl_key}' not found in config['labels'].")

        lbl_meta = labels_cfg[lbl_key]
        lbl_path = ROOT / Path(lbl_meta["file"])
        lbl_col = lbl_meta["label_column"]

        df_lbl = pd.read_csv(lbl_path)
        if "date" not in df_lbl.columns:
            raise ValueError(f"Label file for {lbl_key} must contain a 'date' column.")
        df_lbl["date"] = pd.to_datetime(df_lbl["date"], format="%Y-%m-%d")
        df_lbl = df_lbl.sort_values("date").reset_index(drop=True)

        keep_cols = ["date", lbl_col]

        if lbl_key == "L1_Q":
            frc = lbl_meta.get("future_return_column", None)
            if frc and frc in df_lbl.columns:
                keep_cols.append(frc)

        if lbl_key == "L2":
            for c in ["MA_fast", "MA_slow", "MA_slow_slope"]:
                if c in df_lbl.columns:
                    keep_cols.append(c)

        df_lbl = df_lbl[keep_cols]
        df = df.merge(df_lbl, on="date", how="left", validate="one_to_one")

    return df

modeling_df = load_features_labels(cfg, label_keys=["L1_Q", "L1_extreme", "L2"])
print("modeling_df shape:", modeling_df.shape)


F_CORE = cfg["features"]["F_CORE"]["columns"]
LABEL_COL = cfg["labels"]["L1_Q"]["label_column"]  # y_L1_Q

print("F_CORE:", F_CORE)
print("LABEL:", LABEL_COL)



def get_v1_boundaries(cfg: dict) -> dict[str, tuple[pd.Timestamp, pd.Timestamp]]:
    v = cfg.get("validation", {})
    v1 = v.get("V1", None) or v.get("v1", None)
    if isinstance(v1, dict) and all(k in v1 for k in ["train", "val", "test"]):
        def _pair(x): return (pd.to_datetime(x[0]), pd.to_datetime(x[1]))
        return {k: _pair(v1[k]) for k in ["train", "val", "test"]}

    return {
        "train": (pd.Timestamp("2000-01-03"), pd.Timestamp("2015-12-31")),
        "val":   (pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
        "test":  (pd.Timestamp("2020-01-01"), pd.Timestamp("2025-11-25")),
    }

v1_bounds = get_v1_boundaries(cfg)

def mask_between(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    return (df["date"] >= start) & (df["date"] <= end)

v1_masks = {k: mask_between(modeling_df, *v1_bounds[k]) for k in ["train", "val", "test"]}

V2_FOLDS = [
    {"fold": "F1", "train": ("2000-01-03", "2009-12-31"), "test": ("2010-01-01", "2011-12-31")},
    {"fold": "F2", "train": ("2000-01-03", "2011-12-31"), "test": ("2012-01-01", "2013-12-31")},
    {"fold": "F3", "train": ("2000-01-03", "2013-12-31"), "test": ("2014-01-01", "2015-12-31")},
    {"fold": "F4", "train": ("2000-01-03", "2015-12-31"), "test": ("2016-01-01", "2019-12-31")},
]

def build_v2_masks(df: pd.DataFrame, folds: list[dict]) -> dict[str, dict[str, pd.Series]]:
    out = {}
    for f in folds:
        tr0, tr1 = pd.to_datetime(f["train"][0]), pd.to_datetime(f["train"][1])
        te0, te1 = pd.to_datetime(f["test"][0]), pd.to_datetime(f["test"][1])
        out[f["fold"]] = {
            "train": mask_between(df, tr0, tr1),
            "test":  mask_between(df, te0, te1),
        }
    return out

v2_masks = build_v2_masks(modeling_df, V2_FOLDS)



def extract_xy(df: pd.DataFrame, mask: pd.Series, feat_cols: list[str], label_col: str, dropna: bool=True):
    sub = df.loc[mask].copy()
    cols_to_check = feat_cols + [label_col]
    if dropna:
        ok = sub[cols_to_check].notna().all(axis=1)
        sub = sub.loc[ok].copy()
    X = sub[feat_cols].to_numpy()
    y = sub[label_col].to_numpy()
    return sub, X, y


def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", path)

def plot_confmat(cm: np.ndarray, labels: list, title: str, outpath: Path):
    fig = plt.figure(figsize=(5.2, 4.6))
    ax = fig.add_subplot(111)
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    # annotate counts
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    savefig(outpath)


def plot_timeline(v1_bounds: dict, v2_folds: list[dict], outpath: Path):
    rows = []
    # V1 rows
    for k in ["train", "val", "test"]:
        s, e = v1_bounds[k]
        rows.append({"row": f"V1-{k}", "start": s, "end": e})
    # V2 rows
    for f in v2_folds:
        rows.append({"row": f'V2-{f["fold"]}-train', "start": pd.to_datetime(f["train"][0]), "end": pd.to_datetime(f["train"][1])})
        rows.append({"row": f'V2-{f["fold"]}-test',  "start": pd.to_datetime(f["test"][0]),  "end": pd.to_datetime(f["test"][1])})

    df_t = pd.DataFrame(rows).sort_values(["row"]).reset_index(drop=True)

    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(111)

    y_positions = np.arange(len(df_t))
    for i, r in df_t.iterrows():
        ax.barh(
            y=i,
            width=(r["end"] - r["start"]).days,
            left=r["start"],
            height=0.6
        )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(df_t["row"])
    ax.set_title("Week 6 validation design timeline (V1 + V2 folds)")
    ax.set_xlabel("Date")
    savefig(outpath)

plot_timeline(v1_bounds, V2_FOLDS, OUT_DIR / "W6-F1_timeline_V1_V2.png")



def plot_label_distribution_v1(df: pd.DataFrame, v1_masks: dict, feat_cols: list[str], label_col: str, outpath: Path):
    order = [-1.0, 0.0, 1.0]
    splits = ["train", "val", "test"]
    counts = {s: None for s in splits}

    for s in splits:
        sub, _, y = extract_xy(df, v1_masks[s], feat_cols, label_col, dropna=True)
        vc = pd.Series(y).value_counts()
        counts[s] = [int(vc.get(k, 0)) for k in order]

    x = np.arange(len(order))
    width = 0.25

    fig = plt.figure(figsize=(8.5, 4.8))
    ax = fig.add_subplot(111)
    for i, s in enumerate(splits):
        ax.bar(x + (i - 1) * width, counts[s], width=width, label=f"V1-{s}")

    ax.set_xticks(x)
    ax.set_xticklabels([str(k) for k in order])
    ax.set_xlabel("Class label (y_L1_Q)")
    ax.set_ylabel("Count (after dropna)")
    ax.set_title("Week 6 — L1_Q class distribution by V1 split (after dropna)")
    ax.legend()
    savefig(outpath)

plot_label_distribution_v1(modeling_df, v1_masks, F_CORE, LABEL_COL, OUT_DIR / "W6-F2_label_distribution_V1.png")


def plot_dropna_impact_v1(df: pd.DataFrame, v1_masks: dict, feat_cols: list[str], label_col: str, outpath: Path):
    splits = ["train", "val", "test"]
    totals, valids, dropped = [], [], []

    for s in splits:
        sub_all = df.loc[v1_masks[s]]
        total = len(sub_all)
        cols_to_check = feat_cols + [label_col]
        valid = int(sub_all[cols_to_check].notna().all(axis=1).sum())
        totals.append(total)
        valids.append(valid)
        dropped.append(total - valid)

    x = np.arange(len(splits))
    fig = plt.figure(figsize=(8, 4.8))
    ax = fig.add_subplot(111)

    ax.bar(x - 0.18, totals, width=0.36, label="Before dropna")
    ax.bar(x + 0.18, valids, width=0.36, label="After dropna")

    for i, d in enumerate(dropped):
        ax.text(i, max(totals[i], valids[i]) * 0.95, f"dropped={d}", ha="center", va="top")

    ax.set_xticks(x)
    ax.set_xticklabels([f"V1-{s}" for s in splits])
    ax.set_ylabel("Rows")
    ax.set_title("Week 6 — NaN policy impact (before vs after dropna)")
    ax.legend()
    savefig(outpath)

plot_dropna_impact_v1(modeling_df, v1_masks, F_CORE, LABEL_COL, OUT_DIR / "W6-F3_dropna_impact_V1.png")


LABEL_ORDER = [-1.0, 0.0, 1.0]

def train_eval_v1_E1_E3(df: pd.DataFrame):
    # V1 train/val/test with dropna=True
    sub_tr, X_tr, y_tr = extract_xy(df, v1_masks["train"], F_CORE, LABEL_COL, dropna=True)
    sub_va, X_va, y_va = extract_xy(df, v1_masks["val"],   F_CORE, LABEL_COL, dropna=True)
    sub_te, X_te, y_te = extract_xy(df, v1_masks["test"],  F_CORE, LABEL_COL, dropna=True)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_te)

    # E1 (V1 Logistic)
    clf_e1 = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", multi_class="auto", max_iter=1000)
    clf_e1.fit(X_tr_s, y_tr)
    yhat_va_e1 = clf_e1.predict(X_va_s)
    yhat_te_e1 = clf_e1.predict(X_te_s)

    cm_va_e1 = confusion_matrix(y_va, yhat_va_e1, labels=LABEL_ORDER)
    cm_te_e1 = confusion_matrix(y_te, yhat_te_e1, labels=LABEL_ORDER)

    # E3 (V1 LinearSVC)
    clf_e3 = LinearSVC(C=1.0, class_weight=None, max_iter=5000)
    clf_e3.fit(X_tr_s, y_tr)
    yhat_va_e3 = clf_e3.predict(X_va_s)
    yhat_te_e3 = clf_e3.predict(X_te_s)

    cm_va_e3 = confusion_matrix(y_va, yhat_va_e3, labels=LABEL_ORDER)
    cm_te_e3 = confusion_matrix(y_te, yhat_te_e3, labels=LABEL_ORDER)

    return (cm_va_e1, cm_te_e1, cm_va_e3, cm_te_e3)

cm_va_e1, cm_te_e1, cm_va_e3, cm_te_e3 = train_eval_v1_E1_E3(modeling_df)

plot_confmat(cm_va_e1, LABEL_ORDER, "W6-F4 — V1 Val Confusion Matrix (E1 Logistic)", OUT_DIR / "W6-F4_cm_V1_val_E1.png")
plot_confmat(cm_te_e1, LABEL_ORDER, "W6-F4 — V1 Test Confusion Matrix (E1 Logistic)", OUT_DIR / "W6-F4_cm_V1_test_E1.png")
plot_confmat(cm_va_e3, LABEL_ORDER, "W6-F4 — V1 Val Confusion Matrix (E3 LinearSVM)", OUT_DIR / "W6-F4_cm_V1_val_E3.png")
plot_confmat(cm_te_e3, LABEL_ORDER, "W6-F4 — V1 Test Confusion Matrix (E3 LinearSVM)", OUT_DIR / "W6-F4_cm_V1_test_E3.png")

def train_eval_v2_E2(df: pd.DataFrame, fold_name: str):
    m_tr = v2_masks[fold_name]["train"]
    m_te = v2_masks[fold_name]["test"]

    _, X_tr, y_tr = extract_xy(df, m_tr, F_CORE, LABEL_COL, dropna=True)
    _, X_te, y_te = extract_xy(df, m_te, F_CORE, LABEL_COL, dropna=True)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    # E2 (V2 Logistic; Week 6 Step2 settings)
    clf = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", multi_class="multinomial", max_iter=200, class_weight=None)
    clf.fit(X_tr_s, y_tr)
    yhat_te = clf.predict(X_te_s)

    cm_te = confusion_matrix(y_te, yhat_te, labels=LABEL_ORDER)
    return cm_te

cm_f1_e2 = train_eval_v2_E2(modeling_df, "F1")
cm_f2_e2 = train_eval_v2_E2(modeling_df, "F2")

plot_confmat(cm_f1_e2, LABEL_ORDER, "W6-F4 — V2 F1 Test Confusion Matrix (E2 Logistic)", OUT_DIR / "W6-F4_cm_V2_F1_test_E2.png")
plot_confmat(cm_f2_e2, LABEL_ORDER, "W6-F4 — V2 F2 Test Confusion Matrix (E2 Logistic)", OUT_DIR / "W6-F4_cm_V2_F2_test_E2.png")


TABLES_DIR = ROOT / "results" / "tables"
paths = {
    "E1": TABLES_DIR / "metrics_E1.csv",
    "E3": TABLES_DIR / "metrics_E3.csv",
    "E2": TABLES_DIR / "metrics_E2.csv",
    "E4": TABLES_DIR / "metrics_E4.csv",
}

for k, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing metrics file: {p}")

m1 = pd.read_csv(paths["E1"])
m3 = pd.read_csv(paths["E3"])
m2 = pd.read_csv(paths["E2"])
m4 = pd.read_csv(paths["E4"])

# V1 test rows
e1_v1_test = m1.loc[m1["split"] == "test"].iloc[0]
e3_v1_test = m3.loc[m3["split"] == "test"].iloc[0]

# V2 mean test rows
e2_v2_test_mean = m2.loc[m2["split"] == "test", ["accuracy", "macro_f1"]].mean()
e4_v2_test_mean = m4.loc[m4["split"] == "test", ["accuracy", "macro_f1"]].mean()

comp = pd.DataFrame([
    {"setting": "V1 test", "model": "Logistic (E1)", "accuracy": e1_v1_test["accuracy"], "macro_f1": e1_v1_test["macro_f1"]},
    {"setting": "V1 test", "model": "LinearSVM (E3)", "accuracy": e3_v1_test["accuracy"], "macro_f1": e3_v1_test["macro_f1"]},
    {"setting": "V2 mean test", "model": "Logistic (E2)", "accuracy": e2_v2_test_mean["accuracy"], "macro_f1": e2_v2_test_mean["macro_f1"]},
    {"setting": "V2 mean test", "model": "LinearSVM (E4)", "accuracy": e4_v2_test_mean["accuracy"], "macro_f1": e4_v2_test_mean["macro_f1"]},
])

def plot_v1_v2_comparison(df_comp: pd.DataFrame, outpath: Path):
    labels = [f'{r["setting"]}\n{r["model"]}' for _, r in df_comp.iterrows()]
    x = np.arange(len(labels))

    fig = plt.figure(figsize=(12, 4.8))
    ax = fig.add_subplot(111)

    ax.bar(x - 0.18, df_comp["accuracy"].to_numpy(), width=0.36, label="Accuracy")
    ax.bar(x + 0.18, df_comp["macro_f1"].to_numpy(), width=0.36, label="Macro-F1")

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.0)
    ax.set_title("Week 6 — V1 test vs V2 mean test performance (from saved metrics CSVs)")
    ax.legend()
    savefig(outpath)

plot_v1_v2_comparison(comp, OUT_DIR / "W6-F5_V1_vs_V2_test_metrics.png")

print("Done. Figures saved under:", OUT_DIR)


ROOT: C:\Users\simon\Desktop\my thesis\ThesisProject
CFG : C:\Users\simon\Desktop\my thesis\ThesisProject\config\labels_features.yaml
OUT : C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6
modeling_df shape: (6752, 22)
F_CORE: ['f_r_lag1', 'f_past_ret_5', 'f_past_ret_20', 'f_vol_10', 'f_vol_30']
LABEL: y_L1_Q
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F1_timeline_V1_V2.png
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F2_label_distribution_V1.png
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F3_dropna_impact_V1.png
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F4_cm_V1_val_E1.png


c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F4_cm_V1_test_E1.png
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F4_cm_V1_val_E3.png
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F4_cm_V1_test_E3.png
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F4_cm_V2_F1_test_E2.png


c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F4_cm_V2_F2_test_E2.png
Saved: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6\W6-F5_V1_vs_V2_test_metrics.png
Done. Figures saved under: C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\week6


[OK] Saved Week 5 thesis figures to: C:\Users\simon\Desktop\my thesis\ThesisProject\results\week05_figures


SystemExit: 0

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
